In [105]:
# !pip install catboost

# Import Libraries

In [106]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score,
    matthews_corrcoef,
    f1_score,
    confusion_matrix,
    brier_score_loss
)

from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load Dataset

In [107]:
engineered = pd.read_csv("/content/DiabeticCKD_engineered.csv")
raw = pd.read_csv("/content/DiabeticCKD_dataset.csv")

print("Engineered:", engineered.shape)
print("Raw:", raw.shape)

Engineered: (4000, 34)
Raw: (4000, 26)


# Match index to patient ID

In [108]:
engineered.reset_index(inplace=True)
raw.reset_index(inplace=True)

patient_lookup = raw[
    ["index", "Patient ID"]
].copy()

df = engineered.merge(
    patient_lookup,
    on="index",
    how="left"
)

print(df[["index", "Patient ID"]].head())
print("Missing Patient IDs:", df["Patient ID"].isna().sum())

   index  Patient ID
0      0           1
1      1           1
2      2           1
3      3           1
4      4           1
Missing Patient IDs: 0


In [109]:
df.columns

Index(['index', 'Gender', 'Job', 'Family_Background_of_Diabetes', 'Height',
       'Diabetic_Year', 'Age', 'Average_Age', 'Weight', 'Average_Weight',
       'BMI', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes',
       'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep',
       'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly',
       'Urination_Properly', 'Urinary_Infection', 'Pain_killer',
       'Calorie_Intake', 'CKD', 'Current_BMI', 'BMI_Diabetes', 'Age_Diabetes',
       'Weight_Diff', 'Age_Diff', 'BMI_per_Year', 'Lifestyle_Score',
       'Risk_Score', 'Treatment_Score', 'Patient ID'],
      dtype='object')

# Define X and y

In [110]:
TARGET = "CKD"
DROP_COLUMNS = [
    "CKD",
    "index",
    "Patient ID",
    "Average_Age",
    "Average_Weight",
    "BMI",
    "Height",
    "Weight"
]

DROP_COLUMNS = [
    c for c in DROP_COLUMNS
    if c in df.columns
]

X = df.drop(columns=DROP_COLUMNS)
y = df[TARGET].astype(int)

print("X:", X.shape)
print("y:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X: (4000, 28)
y: (4000,)

Features:
['Gender', 'Job', 'Family_Background_of_Diabetes', 'Diabetic_Year', 'Age', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes', 'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep', 'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly', 'Urination_Properly', 'Urinary_Infection', 'Pain_killer', 'Calorie_Intake', 'Current_BMI', 'BMI_Diabetes', 'Age_Diabetes', 'Weight_Diff', 'Age_Diff', 'BMI_per_Year', 'Lifestyle_Score', 'Risk_Score', 'Treatment_Score']


# categorical columns for SMOTENC

In [111]:
categorical_columns = [
    "Gender",
    "Job",
    "Family_Background_of_Diabetes",
    "Follow_suggested_Diet",
    "Take_Medicine_for_Diabetes",
    "Take_Insulin",
    "Hypertension",
    "Heart_Disease",
    "Sleep",
    "Water_Consumption",
    "Smoke",
    "Zarda_Betel_Leaf",
    "Walk_Regularly",
    "Urination_Properly",
    "Urinary_Infection",
    "Pain_killer"
]

categorical_columns = [c for c in categorical_columns if c in X.columns]
categorical_indices = [X.columns.get_loc(c) for c in categorical_columns]

print(categorical_columns)
print(categorical_indices)

['Gender', 'Job', 'Family_Background_of_Diabetes', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes', 'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep', 'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly', 'Urination_Properly', 'Urinary_Infection', 'Pain_killer']
[0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


# Strongest models

In [112]:
STRONGEST_MODELS = [
    "LightGBM",
    "XGBoost",
    "Random_Forest"
]

MODEL_FILES = {
    "LightGBM": sorted(glob.glob("/content/models/LightGBM/fold*.joblib")),
    "XGBoost": sorted(glob.glob("/content/models/XGBoost/fold*.joblib")),
    "Random_Forest": sorted(glob.glob("/content/models/Random_Forest/fold*.joblib"))
}

for name, files in MODEL_FILES.items():
    print(name, len(files))
    print(files)

LightGBM 5
['/content/models/LightGBM/fold1.joblib', '/content/models/LightGBM/fold2.joblib', '/content/models/LightGBM/fold3.joblib', '/content/models/LightGBM/fold4.joblib', '/content/models/LightGBM/fold5.joblib']
XGBoost 5
['/content/models/XGBoost/fold1.joblib', '/content/models/XGBoost/fold2.joblib', '/content/models/XGBoost/fold3.joblib', '/content/models/XGBoost/fold4.joblib', '/content/models/XGBoost/fold5.joblib']
Random_Forest 5
['/content/models/Random_Forest/fold1.joblib', '/content/models/Random_Forest/fold2.joblib', '/content/models/Random_Forest/fold3.joblib', '/content/models/Random_Forest/fold4.joblib', '/content/models/Random_Forest/fold5.joblib']


# Load folds

In [113]:
def load_fold(path):
    obj = joblib.load(path)

    required = [
        "fold",
        "best_pipeline",
        "best_params",
        "threshold",
        "train_indices",
        "test_indices",
        "inner_best_score",
        "y_true",
        "y_prob",
        "y_pred"
    ]

    missing = [k for k in required if k not in obj]

    if missing:
        raise KeyError(f"{path} is missing: {missing}")

    return obj

In [114]:
test_fold = load_fold(MODEL_FILES["LightGBM"][0])
print(test_fold.keys())

dict_keys(['fold', 'best_pipeline', 'best_params', 'threshold', 'train_indices', 'test_indices', 'inner_best_score', 'y_true', 'y_prob', 'y_pred'])


# Get base estimator

In [115]:
def get_base_estimator(best_pipeline):
    # sklearn Pipeline
    if hasattr(best_pipeline,"steps"):
        return clone(
            best_pipeline.steps[-1][1]
        )

    return clone(best_pipeline)

# Calculate metrics

In [116]:
def calculate_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "PR_AUC": average_precision_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Brier_Score": brier_score_loss(y_true, y_prob)
    }

# Select threshold

In [117]:
def select_threshold(y_true, y_prob):
    thresholds = np.linspace(
        0.05,
        0.95,
        91
    )

    best_threshold = 0.5
    best_mcc = -np.inf

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue

        mcc = matthews_corrcoef( y_true, y_pred)
        if mcc > best_mcc:
            best_mcc = mcc
            best_threshold = threshold

    return best_threshold

# Get inner threshold

In [118]:
def inner_threshold(model, X_train, y_train, groups_train):

    inner_cv = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof_prob = np.zeros(len(y_train))

    for inner_train_idx, inner_valid_idx in inner_cv.split(X_train, y_train, groups=groups_train):

        X_inner_train = X_train.iloc[inner_train_idx]
        y_inner_train = y_train.iloc[inner_train_idx]
        X_inner_valid = X_train.iloc[inner_valid_idx]

        model_inner = clone(model)
        model_inner.fit(X_inner_train, y_inner_train)
        oof_prob[inner_valid_idx] = model_inner.predict_proba(X_inner_valid)[:, 1]

    return select_threshold(y_train.values, oof_prob)

# Class Weighting function

In [119]:
def apply_class_weight_model(base_model, y_train):

    model = clone(base_model)
    model_name = model.__class__.__name__

    ratio = ((y_train == 0).sum() / (y_train == 1).sum())

    if model_name in ["LGBMClassifier", "XGBClassifier"]:
        model.set_params(scale_pos_weight=ratio)

    elif model_name == "RandomForestClassifier":
        model.set_params(class_weight="balanced")

    else:
        params = model.get_params()
        if "class_weight" in params:
            model.set_params(class_weight="balanced")

    return model

# SMOTENC function

In [120]:
def make_smotenc_model(base_model):
    sampler = SMOTENC(
        categorical_features=categorical_indices,
        random_state=RANDOM_STATE,
        k_neighbors=5
    )

    return ImbPipeline([
        ("smote", sampler),
        ("model", clone(base_model))
    ])

# Apply imbalance handling

In [121]:
experiment2_results = []

for model_name in STRONGEST_MODELS:

    print(f"\n===== {model_name} =====")

    for fold_number, fold_file in enumerate(MODEL_FILES[model_name], start=1):

        fold = load_fold(fold_file)
        train_idx = np.asarray(fold["train_indices"])
        test_idx = np.asarray(fold["test_indices"])

        X_train = X.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        groups_train = df["Patient ID"].iloc[train_idx].values

        base_model = get_base_estimator(fold["best_pipeline"])

        # ------------------------------------------
        # 1. NO ADJUSTMENT
        # ------------------------------------------

        model = clone(base_model)

        # Use the EXISTING Experiment 1 threshold
        threshold = fold["threshold"]

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = calculate_metrics(y_test.values, y_prob, threshold)

        experiment2_results.append({
            "Model": model_name,
            "Adjustment": "None",
            "Fold": fold_number,
            "Threshold": threshold,
            **metrics
        })

        # ------------------------------------------
        # 2. CLASS WEIGHTING
        # ------------------------------------------

        model = apply_class_weight_model(base_model, y_train)
        threshold = inner_threshold(model, X_train, y_train, groups_train)

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = calculate_metrics(y_test.values, y_prob, threshold)

        experiment2_results.append({
            "Model": model_name,
            "Adjustment": "Class Weighting",
            "Fold": fold_number,
            "Threshold": threshold,
            **metrics
        })

        # ------------------------------------------
        # 3. SMOTENC
        # ------------------------------------------

        model = make_smotenc_model(base_model)
        threshold = inner_threshold(model, X_train, y_train, groups_train)

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = calculate_metrics(y_test.values, y_prob, threshold)

        experiment2_results.append({
            "Model": model_name,
            "Adjustment": "SMOTENC",
            "Fold": fold_number,
            "Threshold": threshold,
            **metrics
        })

        print(
            f"Fold {fold_number} complete"
        )


===== LightGBM =====
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete

===== XGBoost =====
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete

===== Random_Forest =====
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [122]:
experiment2_results = pd.DataFrame(experiment2_results)

experiment2_results.to_csv(
    "/content/experiment2_dataset2_results.csv",
    index=False,
    encoding="utf-8-sig"
)

metric_columns = [
            "PR_AUC",
            "MCC",
            "F1",
            "Sensitivity",
            "Specificity",
            "Brier_Score"
        ]

experiment2_grouped = (experiment2_results.groupby(["Model", "Adjustment"])[metric_columns].agg(["mean", "std"]))
experiment2_summary = pd.DataFrame(index=experiment2_grouped.index)

for metric in metric_columns:

        mean_values = experiment2_grouped[(metric, "mean")]
        std_values = experiment2_grouped[(metric, "std")]

        experiment2_summary[metric] = [
            (
                f"{mean:.4f} ± {std:.4f}"
                if pd.notna(mean) and pd.notna(std)
                else (
                    f"{mean:.4f} ± NaN"
                    if pd.notna(mean)
                    else "NA"
                )
            )
            for mean, std in zip(mean_values, std_values)
        ]

experiment2_summary.reset_index(inplace=True)

experiment2_summary.to_csv(
    "/content/experiment2_dataset2_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(experiment2_summary)

,Model,Adjustment,PR_AUC,MCC,F1,Sensitivity,Specificity,Brier_Score
0,LightGBM,Class Weighting,0.4371 ± 0.0358,0.4055 ± 0.0460,0.4604 ± 0.0461,0.4638 ± 0.0750,0.9415 ± 0.0188,0.1038 ± 0.0291
1,LightGBM,None,0.4401 ± 0.0397,0.3858 ± 0.0488,0.4410 ± 0.0420,0.5290 ± 0.1681,0.9049 ± 0.0692,0.0864 ± 0.0129
2,LightGBM,SMOTENC,0.4243 ± 0.0521,0.3665 ± 0.0200,0.4109 ± 0.0160,0.3864 ± 0.1034,0.9483 ± 0.0332,0.0879 ± 0.0154
3,Random_Forest,Class Weighting,0.5041 ± 0.0483,0.4217 ± 0.0227,0.4663 ± 0.0223,0.6790 ± 0.1263,0.8646 ± 0.0572,0.1332 ± 0.0131
4,Random_Forest,None,0.5181 ± 0.0331,0.4202 ± 0.0338,0.4702 ± 0.0293,0.4551 ± 0.0757,0.9492 ± 0.0197,0.0635 ± 0.0054
5,Random_Forest,SMOTENC,0.3478 ± 0.0841,0.3087 ± 0.0519,0.3550 ± 0.0613,0.7244 ± 0.1921,0.7330 ± 0.1506,0.1416 ± 0.0220
6,XGBoost,Class Weighting,0.4633 ± 0.0381,0.3937 ± 0.0333,0.4525 ± 0.0295,0.5141 ± 0.0769,0.9169 ± 0.0394,0.0845 ± 0.0089
7,XGBoost,None,0.4654 ± 0.0340,0.3890 ± 0.0597,0.4422 ± 0.0580,0.6183 ± 0.1279,0.8717 ± 0.0622,0.0758 ± 0.0105
8,XGBoost,SMOTENC,0.4343 ± 0.0302,0.3833 ± 0.0436,0.4363 ± 0.0397,0.5294 ± 0.1618,0.9042 ± 0.0537,0.0923 ± 0.0165


# Added value of lifestyle and comorbidity variables


In [123]:
AGE_DURATION_COLUMNS = ["Age", "Diabetic_Year"]
LIFESTYLE_COLUMNS = [
    "Follow_suggested_Diet",
    "Sleep",
    "Water_Consumption",
    "Smoke",
    "Zarda_Betel_Leaf",
    "Walk_Regularly"
]

LIFESTYLE_COLUMNS = [
    c for c in LIFESTYLE_COLUMNS
    if c in X.columns
]

FEATURE_SETS = {
    "Age + Diabetic Year": X[AGE_DURATION_COLUMNS].copy(),
    "Complete": X.copy(),
    "Without Lifestyle": X.drop(columns=LIFESTYLE_COLUMNS).copy()
}

for name, xdata in FEATURE_SETS.items():
    print(name, xdata.shape)

Age + Diabetic Year (4000, 2)
Complete (4000, 28)
Without Lifestyle (4000, 22)


# Get exp-3 results

In [124]:
experiment3_results = []

MODEL_NAME = "LightGBM"

for feature_name, X_features in FEATURE_SETS.items():

    print(f"\n===== {feature_name} =====")
    for fold_number, fold_file in enumerate(MODEL_FILES[MODEL_NAME], start=1):

        fold = load_fold(fold_file)
        train_idx = np.asarray(fold["train_indices"])
        test_idx = np.asarray(fold["test_indices"])

        X_train = X_features.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()
        X_test = X_features.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        groups_train = df["Patient ID"].iloc[train_idx].values
        base_model = get_base_estimator(fold["best_pipeline"])

        # ------------------------------------------
        # Each feature set gets its own threshold
        # ------------------------------------------

        threshold = inner_threshold(
            base_model,
            X_train,
            y_train,
            groups_train
        )

        # ------------------------------------------
        # Fit outer training fold
        # ------------------------------------------

        model = clone(base_model)
        model.fit(X_train, y_train)

        # ------------------------------------------
        # Outer test prediction
        # ------------------------------------------

        y_prob = model.predict_proba(X_test)[:, 1]
        metrics = calculate_metrics(y_test.values, y_prob, threshold)

        experiment3_results.append({
            "Feature_Set": feature_name,
            "Fold": fold_number,
            "Threshold": threshold,
            **metrics
        })

        print(
            f"Fold {fold_number}: "
            f"PR AUC = {metrics['PR_AUC']:.4f}"
        )


===== Age + Diabetic Year =====
Fold 1: PR AUC = 0.2166
Fold 2: PR AUC = 0.1553
Fold 3: PR AUC = 0.2421
Fold 4: PR AUC = 0.1885
Fold 5: PR AUC = 0.1381

===== Complete =====
Fold 1: PR AUC = 0.4686
Fold 2: PR AUC = 0.4198
Fold 3: PR AUC = 0.4826
Fold 4: PR AUC = 0.3832
Fold 5: PR AUC = 0.4462

===== Without Lifestyle =====
Fold 1: PR AUC = 0.4695
Fold 2: PR AUC = 0.4421
Fold 3: PR AUC = 0.4682
Fold 4: PR AUC = 0.4353
Fold 5: PR AUC = 0.4316


# Save exp-3 results

In [125]:
experiment3_results = pd.DataFrame(experiment3_results)

experiment3_results.to_csv(
    "/content/experiment3_dataset2_results.csv",
    index=False,
    encoding="utf-8-sig"
)

metric_columns = [
            "PR_AUC",
            "MCC",
            "F1",
            "Sensitivity",
            "Specificity",
            "Brier_Score"
        ]
experiment3_grouped = (experiment3_results.groupby("Feature_Set")[metric_columns].agg(["mean", "std"]))
experiment3_summary = pd.DataFrame(index=experiment3_grouped.index)

for metric in metric_columns:

        mean_values = experiment3_grouped[(metric, "mean")]
        std_values = experiment3_grouped[(metric, "std")]

        experiment3_summary[metric] = [
            (
                f"{mean:.4f} ± {std:.4f}"
                if pd.notna(mean) and pd.notna(std)
                else (
                    f"{mean:.4f} ± NaN"
                    if pd.notna(mean)
                    else "NA"
                )
            )
            for mean, std in zip(mean_values, std_values)
        ]

experiment3_summary.reset_index(inplace=True)
experiment3_summary.to_csv(
    "/content/experiment3_dataset2_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(experiment3_summary)

,Feature_Set,PR_AUC,MCC,F1,Sensitivity,Specificity,Brier_Score
0,Age + Diabetic Year,0.1881 ± 0.0427,0.1539 ± 0.0685,0.2350 ± 0.0326,0.6736 ± 0.2324,0.5824 ± 0.1334,0.1398 ± 0.0658
1,Complete,0.4401 ± 0.0397,0.3826 ± 0.0478,0.4408 ± 0.0445,0.4469 ± 0.0696,0.9392 ± 0.0106,0.0864 ± 0.0129
2,Without Lifestyle,0.4493 ± 0.0182,0.3990 ± 0.0532,0.4533 ± 0.0504,0.4689 ± 0.1049,0.9379 ± 0.0197,0.0850 ± 0.0140
